# Analiza histogramowa i boxplotowa pierwszego utworzonego zbioru
W tym notatniku wczytujemy pierwszy utworzony zbiór po imputacji i skalowaniu, a następnie wykonujemy analizę rozkładów cech numerycznych za pomocą histogramów i wykresów pudełkowych.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["axes.labelsize"] = 10

In [ ]:
data_dir = Path("./data")

# Wpisz nazwe konkretnego pliku dla tego notatnika
dataset_filename = "loan_data_simple_b_minmax.csv"

selected_dataset_path = data_dir / dataset_filename
if not selected_dataset_path.exists():
    available_files = sorted([p.name for p in data_dir.glob("loan_data_*_*.csv")])
    raise FileNotFoundError(
        f"Nie znaleziono pliku: {selected_dataset_path}\nDostepne pliki: {available_files}"
    )

print(f"Wczytywany zbior: {selected_dataset_path}")
df = pd.read_csv(selected_dataset_path)
display(df.head())

In [ ]:
print("Ksztalt danych:", df.shape)
print("\nTypy kolumn:")
display(df.dtypes.to_frame("dtype"))

print("\nBraki danych:")
display(df.isna().sum().to_frame("missing_count"))

num_cols = [
    col for col in df.select_dtypes(include=[np.number]).columns.tolist()
    if col != "loan_status"
]
print(f"\nLiczba kolumn numerycznych (bez loan_status): {len(num_cols)}")
print("Kolumny numeryczne:", num_cols)

In [ ]:
# Analiza histogramowa wszystkich kolumn numerycznych
n_cols = 3
n_rows = int(np.ceil(len(num_cols) / n_cols)) if num_cols else 1

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 4 * n_rows))
axes = np.array(axes).reshape(-1)

for i, col in enumerate(num_cols):
    sns.histplot(df[col], bins=30, kde=True, ax=axes[i], color="#1f77b4")
    axes[i].set_title(f"Histogram: {col}")
    axes[i].set_xlabel(col)
    axes[i].set_ylabel("Licznosc")

for j in range(len(num_cols), len(axes)):
    axes[j].axis("off")

plt.suptitle("Rozklady cech numerycznych (histogram + KDE)", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Analiza boxplotowa wszystkich kolumn numerycznych
if num_cols:
    plot_df = df[num_cols].melt(var_name="cecha", value_name="wartosc")
    
    plt.figure(figsize=(14, max(6, 0.5 * len(num_cols))))
    sns.boxplot(data=plot_df, y="cecha", x="wartosc", orient="h", color="#ff7f0e")
    plt.title("Boxploty cech numerycznych")
    plt.xlabel("Wartosc")
    plt.ylabel("Cecha")
    plt.tight_layout()
    plt.show()
else:
    print("Brak kolumn numerycznych do analizy boxplotowej.")

In [ ]:
# Porownanie histogramow i boxplotow dla wybranych cech
top_features = df[num_cols].var(numeric_only=True).sort_values(ascending=False).head(4).index.tolist() if num_cols else []

if top_features:
    fig, axes = plt.subplots(len(top_features), 2, figsize=(14, 4 * len(top_features)))
    if len(top_features) == 1:
        axes = np.array([axes])

    for i, feature in enumerate(top_features):
        sns.histplot(df[feature], bins=30, kde=True, ax=axes[i, 0], color="#2ca02c")
        axes[i, 0].set_title(f"Histogram: {feature}")
        axes[i, 0].set_xlabel(feature)
        axes[i, 0].set_ylabel("Licznosc")

        sns.boxplot(x=df[feature], ax=axes[i, 1], color="#d62728")
        axes[i, 1].set_title(f"Boxplot: {feature}")
        axes[i, 1].set_xlabel(feature)

    plt.suptitle("Porownanie histogramu i boxplotu dla cech o najwiekszej wariancji", y=1.01)
    plt.tight_layout()
    plt.show()
else:
    print("Brak kolumn numerycznych do porownania histogramow i boxplotow.")